# Prompt Baseline

In [1]:
# === ENVIRONMENT & FILEPATH SETUP ===
import os
import sys
import ctypes

try:
    ctypes.CDLL("/home/emmy/miniconda3/envs/mlbio_gpu/lib/python3.10/site-packages/nvidia/cu13/lib/libnvJitLink.so.13")
    ctypes.CDLL("/home/emmy/miniconda3/envs/mlbio_gpu/lib/python3.10/site-packages/nvidia/nccl/lib/libnccl.so.2")
except Exception:
    pass

codebase_path = "../"
if codebase_path not in sys.path:
    sys.path.insert(0, codebase_path)

for key in list(sys.modules.keys()):
    if key.startswith("src"):
        del sys.modules[key]

DATA_DIR = f"{codebase_path}/output/cache"
ENV_PATH = f"{codebase_path}/artifacts/.env"
MODELS_DIR = f"{codebase_path}/output/models"
ARTIFACTS_DIR = f"{codebase_path}/artifacts"
print("💻 Local Lab Server Environment Loaded.")
cuda_link_path = "/home/emmy/miniconda3/envs/mlbio_gpu/lib/python3.10/site-packages/nvidia/cu13/lib"
nccl_link_path = "/home/emmy/miniconda3/envs/mlbio_gpu/lib/python3.10/site-packages/nvidia/nccl/lib"
os.environ["LD_LIBRARY_PATH"] = os.environ.get("LD_LIBRARY_PATH", "") + ":" + cuda_link_path + ":" + nccl_link_path
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"


💻 Local Lab Server Environment Loaded.


## 1. Load Data & Base Model

In [2]:
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from src.inference.evaluate import run_evaluation

def load_jsonl(path):
    with open(path, 'r', encoding='utf-8') as f:
        return [json.loads(line) for line in f]

val_full = load_jsonl(f"{DATA_DIR}/val_full_info.jsonl")
val_struct = load_jsonl(f"{DATA_DIR}/val_structural.jsonl")
print(f"Loaded {len(val_full)} full-info validation samples and {len(val_struct)} structural validation samples.")

W0619 02:00:11.179000 1418520 site-packages/torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


W0619 02:00:11.190000 1418520 site-packages/torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Loaded 2039 full-info validation samples and 2039 structural validation samples.


In [3]:
# === SELECT MODEL ===
MODEL_ID = "ibm-granite/granite-3.3-2b-instruct"

In [4]:
# === LOAD MODEL ===
COMPUTE_DTYPE = torch.bfloat16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=False,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    bnb_4bit_use_double_quant=True
)

if MODEL_ID == "mistralai/Mistral-Nemo-Instruct-2407":
	tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, 
											trust_remote_code=True,
											fix_mistral_regex=True
											)
else:
	tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, 
											trust_remote_code=True,
											)
 
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    dtype=COMPUTE_DTYPE,
)
model.eval()
print("Model loaded successfully!")

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/home/emmy/miniconda3/envs/mlbio_unsloth/lib/python3.11/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Model loaded successfully!


## 2. Evaluate Baseline Full Information Prompt

In [5]:
# Evaluate on Full Info Dataset
acc_full, results_full = run_evaluation(
    model=model,
    tokenizer=tokenizer,
    dataset=val_full,
    training_strategy="baseline",
    prompt_format="FullInfo",
    model_name=MODEL_ID,
    output_csv=f"{ARTIFACTS_DIR}/experiment_summary.csv"
)


Evaluating baseline - FullInfo:   0%|          | 0/2039 [00:00<?, ?it/s]

/home/emmy/miniconda3/envs/mlbio_unsloth/lib/python3.11/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Evaluating baseline - FullInfo:   0%|          | 1/2039 [00:00<11:38,  2.92it/s]

Evaluating baseline - FullInfo:   0%|          | 2/2039 [00:00<07:40,  4.42it/s]

Evaluating baseline - FullInfo:   0%|          | 3/2039 [00:00<08:22,  4.05it/s]

Evaluating baseline - FullInfo:   0%|          | 4/2039 [00:00<06:46,  5.00it/s]

...

Evaluating baseline - FullInfo: 100%|█████████▉| 2037/2039 [05:01<00:00,  7.05it/s]

Evaluating baseline - FullInfo: 100%|█████████▉| 2038/2039 [05:02<00:00,  6.84it/s]

Evaluating baseline - FullInfo: 100%|██████████| 2039/2039 [05:02<00:00,  6.60it/s]

Evaluating baseline - FullInfo: 100%|██████████| 2039/2039 [05:02<00:00,  6.75it/s]

\n--- Evaluation Results ---
Training Strategy: baseline
Prompt Format: FullInfo
Model: ibm-granite/granite-3.3-2b-instruct
Accuracy: 0.6297
Format Error Rate: 0.0000
Semantic Confusion: 0.4813
Option Bias (A): 0.1099
Latency: 302.19 seconds
Detailed predictions saved to: /data220_2/emmy/mlbio/hw4/output/validation/validation_granite-3.3-2b-instruct_FullInfo_baseline.csv


## 3. Evaluate Baseline Structural-Only Prompt

In [6]:
# Evaluate on Structural Only Dataset
acc_struct, results_struct = run_evaluation(
    model=model,
    tokenizer=tokenizer,
    dataset=val_struct,
    training_strategy="baseline",
    prompt_format="structOnly",
    model_name=MODEL_ID,
    output_csv=f"{ARTIFACTS_DIR}/experiment_summary.csv"
)


Evaluating baseline - structOnly:   0%|          | 0/2039 [00:00<?, ?it/s]

/home/emmy/miniconda3/envs/mlbio_unsloth/lib/python3.11/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Evaluating baseline - structOnly:   0%|          | 2/2039 [00:00<03:24,  9.94it/s]

Evaluating baseline - structOnly:   0%|          | 3/2039 [00:00<03:50,  8.83it/s]

Evaluating baseline - structOnly:   0%|          | 4/2039 [00:00<03:41,  9.21it/s]

...

Evaluating baseline - structOnly: 100%|█████████▉| 2037/2039 [03:48<00:00,  9.39it/s]

Evaluating baseline - structOnly: 100%|█████████▉| 2038/2039 [03:48<00:00,  9.16it/s]

Evaluating baseline - structOnly: 100%|██████████| 2039/2039 [03:49<00:00,  9.14it/s]

Evaluating baseline - structOnly: 100%|██████████| 2039/2039 [03:49<00:00,  8.90it/s]

\n--- Evaluation Results ---
Training Strategy: baseline
Prompt Format: structOnly
Model: ibm-granite/granite-3.3-2b-instruct
Accuracy: 0.6680
Format Error Rate: 0.0005
Semantic Confusion: 0.5306
Option Bias (A): 0.0775
Latency: 229.09 seconds
Detailed predictions saved to: /data220_2/emmy/mlbio/hw4/output/validation/validation_granite-3.3-2b-instruct_structOnly_baseline.csv
